# 01d: Descriptive Statistics (Publication Table 1)

**Purpose:** Generate publication-ready Table 1 with sample characteristics

**Dataset:** COMPAS

**Date:** 2025-11-08

---

## Overview

### Purpose
Create Table 1 for manuscript showing:
- Sample characteristics by recidivism outcome
- Means ± SD for continuous variables
- N (%) for categorical variables
- Standardized mean differences (SMD)
- Statistical tests

### Outputs
- table1_publication.csv
- table1_publication.tex (LaTeX)
- table1_publication.xlsx (formatted)

### Runtime: 1-2 minutes

---

In [ ]:
# Setup
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src"))

from statistics.effect_sizes import cohens_d

PROCESSED_DIR = project_root / "data" / "processed"
TABLES_DIR = project_root / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Setup complete")

## 1. Load Data

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / "compas_cleaned.parquet")
target_col = 'two_year_recid'

# Split by outcome
df_no_recid = df[df[target_col] == 0]
df_recid = df[df[target_col] == 1]

print(f"Total: {len(df):,}")
print(f"No recidivism: {len(df_no_recid):,}")
print(f"Recidivism: {len(df_recid):,}")

## 2. Create Table 1

### 2.1 Sample Size

In [ ]:
table1_rows = []

# Sample size
table1_rows.append({
    'Variable': 'N',
    'Overall': f"{len(df):,}",
    'No_Recidivism': f"{len(df_no_recid):,}",
    'Recidivism': f"{len(df_recid):,}",
    'SMD': '',
    'P_Value': ''
})

### 2.2 Demographics (Categorical)

In [ ]:
# Categorical variables
categorical_vars = ['race', 'sex', 'age_cat']

for var in categorical_vars:
    if var not in df.columns:
        continue
    
    # Header row
    table1_rows.append({
        'Variable': f"{var.upper()}",
        'Overall': '',
        'No_Recidivism': '',
        'Recidivism': '',
        'SMD': '',
        'P_Value': ''
    })
    
    # Chi-squared test
    contingency = pd.crosstab(df[var], df[target_col])
    chi2, p_val, _, _ = stats.chi2_contingency(contingency)
    
    # Categories
    for cat in df[var].unique():
        n_overall = (df[var] == cat).sum()
        pct_overall = n_overall / len(df) * 100
        
        n_no_recid = (df_no_recid[var] == cat).sum()
        pct_no_recid = n_no_recid / len(df_no_recid) * 100
        
        n_recid = (df_recid[var] == cat).sum()
        pct_recid = n_recid / len(df_recid) * 100
        
        # Standardized difference in proportions
        p1 = pct_recid / 100
        p2 = pct_no_recid / 100
        smd = (p1 - p2) / np.sqrt((p1*(1-p1) + p2*(1-p2)) / 2)
        
        table1_rows.append({
            'Variable': f"  {cat}",
            'Overall': f"{n_overall} ({pct_overall:.1f}%)",
            'No_Recidivism': f"{n_no_recid} ({pct_no_recid:.1f}%)",
            'Recidivism': f"{n_recid} ({pct_recid:.1f}%)",
            'SMD': f"{smd:.3f}",
            'P_Value': f"{p_val:.4f}" if cat == df[var].unique()[0] else ''
        })

### 2.3 Continuous Variables

In [ ]:
# Continuous variables
continuous_vars = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in continuous_vars:
    continuous_vars.remove(target_col)

for var in continuous_vars:
    # Overall
    mean_overall = df[var].mean()
    std_overall = df[var].std()
    
    # No recidivism
    mean_no_recid = df_no_recid[var].mean()
    std_no_recid = df_no_recid[var].std()
    
    # Recidivism
    mean_recid = df_recid[var].mean()
    std_recid = df_recid[var].std()
    
    # Standardized mean difference
    smd = cohens_d(df_recid[var], df_no_recid[var], pooled=True)
    
    # T-test
    t_stat, p_val = stats.ttest_ind(df_recid[var], df_no_recid[var])
    
    table1_rows.append({
        'Variable': var,
        'Overall': f"{mean_overall:.2f} ± {std_overall:.2f}",
        'No_Recidivism': f"{mean_no_recid:.2f} ± {std_no_recid:.2f}",
        'Recidivism': f"{mean_recid:.2f} ± {std_recid:.2f}",
        'SMD': f"{smd:.3f}",
        'P_Value': f"{p_val:.4f}"
    })

## 3. Create and Export Table

In [ ]:
# Create DataFrame
table1 = pd.DataFrame(table1_rows)

print("Table 1: Sample Characteristics by Recidivism Outcome")
print("="*80)
display(table1)

# Save CSV
table1.to_csv(TABLES_DIR / 'table1_publication.csv', index=False)
print("\n✓ Saved: table1_publication.csv")

# Save LaTeX
latex_table = table1.to_latex(index=False, escape=False)
with open(TABLES_DIR / 'table1_publication.tex', 'w') as f:
    f.write(latex_table)
print("✓ Saved: table1_publication.tex")

# Save Excel (with formatting)
with pd.ExcelWriter(TABLES_DIR / 'table1_publication.xlsx', engine='openpyxl') as writer:
    table1.to_excel(writer, sheet_name='Table1', index=False)
print("✓ Saved: table1_publication.xlsx")

## Summary

**Table 1 Created:**
- ✓ Sample characteristics by outcome
- ✓ Demographics (N, %)
- ✓ Continuous variables (mean ± SD)
- ✓ Standardized mean differences (SMD)
- ✓ Statistical tests (chi-squared, t-tests)

**Interpretation Guidelines:**
- SMD < 0.1: Negligible difference
- SMD 0.1-0.3: Small difference
- SMD 0.3-0.5: Medium difference
- SMD > 0.5: Large difference

**Next:** 02b_feature_engineering.ipynb